In [ ]:
'''
To better understand how far the optimum moved and its effect, here we monitored the movement of the optimum.

Here we set the optimum at the origin initially, then we let the optimum move according to 3 different strategies (move along
all dimension deterministically, move along the diagonal deterministically and move along one dimnesion deterministically).

We then calculated the fitness of the moving optimum relative to its initial position (the origin). A faster decay of the optimum
fitness will indicate a greater movement of optimum.


'''

In [1]:
from __future__ import division
import numpy as np
from scipy import stats
import scipy.spatial as spa
import numpy.random as rnd
import copy
import time
import pandas as pd
import math
import pickle

In [2]:
"""The deterministic movement of optimum. Move along all dimensions every generation with a pre-defined moving STD"""

class Optimum_move1(object):
    '''Define a single population in FGM. Assuming the fixed optimum is at the orgin of the Coordinate System'''

    def __init__(self,nReps, dim, optimum_move_sigma):
        
        self.nReps = nReps
        self.dim = dim  # The dimension of the landscape

        self.initial_soma_optimum = np.zeros((nReps, dim))

        self.soma_optimum = np.zeros((nReps, dim))
        self.opt_move_sigma = optimum_move_sigma

        self.current_step =0  # Will be used to indicate the number of generations that have been simulated.
        self.opt_fitness = 1



    def move_optimum(self):

        soma_opt_mut = np.random.normal(0, self.opt_move_sigma, size = self.nReps*self.dim).reshape(self.nReps, self.dim)

        self.soma_optimum = self.soma_optimum + soma_opt_mut

        euclidean_distance = np.sqrt(np.sum((self.soma_optimum-self.initial_soma_optimum)**2, axis =1))  
        self.opt_fitness = np.exp(-(euclidean_distance**2)/2)

        self.current_step +=1




    def calculate_fitness(self):

        W = self.opt_fitness

        W_mean = np.nanmean(W)
        W_std = np.nanstd(W)
        W_var = np.var(W)

        return [W_mean, W_std, W_var]




    def simulate(self,stepcount):

        fitness = [self.calculate_fitness()]

        start = time.time()

        while self.current_step <= stepcount:
            self.move_optimum()
            fitness.append(self.calculate_fitness())


        colnames = ['OptFitness_Mean','OptFitness_Std','OptFitness_Var']
        fitness = pd.DataFrame(np.array(fitness),columns=colnames)


        end = time.time()
        print 'TOTAL TIME: ',end-start

        return fitness


    def simulateNsave(self,outfile,stepcount,):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount)
        results.to_csv(outfile,index=False)  
        
        return    

In [3]:
"""The deterministic movement of optimum. Move along a straight line (the diagonal of the landscape) with a given moving rate."""


class Optimum_move2(object):
    '''Define a single population in FGM. Assuming the fixed optimum is at the orgin of the Coordinate System'''

    def __init__(self,nReps, dim, optimum_move_rate):
        
        self.nReps = nReps
        self.dim = dim  # The dimension of the landscape

        self.initial_soma_optimum = np.zeros((nReps, dim))

        self.soma_optimum = np.zeros((nReps, dim))
        self.opt_move_rate = optimum_move_rate

        self.current_step =0  # Will be used to indicate the number of generations that have been simulated.
        self.opt_fitness = 1



    def move_optimum(self):

        soma_opt_move = np.ones((self.nReps, self.dim))*self.opt_move_rate
        self.soma_optimum = self.soma_optimum + soma_opt_move

        euclidean_distance = np.sqrt(np.sum((self.soma_optimum-self.initial_soma_optimum)**2, axis =1))  
        self.opt_fitness = np.exp(-(euclidean_distance**2)/2)

        self.current_step +=1




    def calculate_fitness(self):

        W = self.opt_fitness

        W_mean = np.nanmean(W)
        W_std = np.nanstd(W)
        W_var = np.var(W)

        return [W_mean, W_std, W_var]




    def simulate(self,stepcount):

        fitness = [self.calculate_fitness()]

        start = time.time()

        while self.current_step <= stepcount:
            self.move_optimum()
            fitness.append(self.calculate_fitness())


        colnames = ['OptFitness_Mean','OptFitness_Std','OptFitness_Var']
        fitness = pd.DataFrame(np.array(fitness),columns=colnames)


        end = time.time()
        print 'TOTAL TIME: ',end-start

        return fitness


    def simulateNsave(self,outfile,stepcount,):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount)
        results.to_csv(outfile,index=False)  
        
        return

In [4]:
"""The deterministic movement of optimum. Move along one dimension every generation with a pre-defined moving STD"""


class Optimum_move3(object):
    '''Define a single population in FGM. Assuming the fixed optimum is at the orgin of the Coordinate System'''

    def __init__(self,nReps, dim, optimum_move_sigma):
        
        self.nReps = nReps
        self.dim = dim  # The dimension of the landscape

        self.initial_soma_optimum = np.zeros((nReps, dim))

        self.soma_optimum = np.zeros((nReps, dim))

        self.move_dimension = np.array([np.random.randint(0, dim, size = nReps)]).reshape(nReps,1)
        self.opt_move_sigma = optimum_move_sigma

        self.current_step =0  # Will be used to indicate the number of generations that have been simulated.
        self.opt_fitness = 1



    def move_optimum(self):

        rReps1 =np.expand_dims(np.arange(self.nReps),axis=1)
        soma_opt_mut = np.random.normal(0, self.opt_move_sigma, size = self.nReps).reshape(self.nReps, 1)

        new_soma_optimum = self.soma_optimum.copy()
        new_soma_optimum[rReps1, self.move_dimension] = self.soma_optimum.copy()[rReps1, self.move_dimension] + soma_opt_mut
        self.soma_optimum = new_soma_optimum.copy()


        euclidean_distance = np.sqrt(np.sum((self.soma_optimum-self.initial_soma_optimum)**2, axis =1))  
        self.opt_fitness = np.exp(-(euclidean_distance**2)/2)

        self.current_step +=1




    def calculate_fitness(self):

        W = self.opt_fitness

        W_mean = np.nanmean(W)
        W_std = np.nanstd(W)
        W_var = np.var(W)

        return [W_mean, W_std, W_var]




    def simulate(self,stepcount):

        fitness = [self.calculate_fitness()]

        start = time.time()

        while self.current_step <= stepcount:
            self.move_optimum()
            fitness.append(self.calculate_fitness())


        colnames = ['OptFitness_Mean','OptFitness_Std','OptFitness_Var']
        fitness = pd.DataFrame(np.array(fitness),columns=colnames)


        end = time.time()
        print 'TOTAL TIME: ',end-start

        return fitness


    def simulateNsave(self,outfile,stepcount,):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount)
        results.to_csv(outfile,index=False)  
        
        return